In [ ]:
# Spark Session
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("Basic Transformation - I")
    .master("local[*]")
    .getOrCreate()
)

spark

In [ ]:
# Emp Data & Schema

emp_data = [
    ["001","101","John Doe","30","Male","50000","2015-01-01"],
    ["002","101","Jane Smith","25","Female","45000","2016-02-15"],
    ["003","102","Bob Brown","35","Male","55000","2014-05-01"],
    ["004","102","Alice Lee","28","Female","48000","2017-09-30"],
    ["005","103","Jack Chan","40","Male","60000","2013-04-01"],
    ["006","103","Jill Wong","32","Female","52000","2018-07-01"],
    ["007","101","James Johnson","42","Male","70000","2012-03-15"],
    ["008","102","Kate Kim","29","Female","51000","2019-10-01"],
    ["009","103","Tom Tan","33","Male","58000","2016-06-01"],
    ["010","104","Lisa Lee","27","Female","47000","2018-08-01"],
    ["011","104","David Park","38","Male","65000","2015-11-01"],
    ["012","105","Susan Chen","31","Female","54000","2017-02-15"],
    ["013","106","Brian Kim","45","Male","75000","2011-07-01"],
    ["014","107","Emily Lee","26","Female","46000","2019-01-01"],
    ["015","106","Michael Lee","37","Male","63000","2014-09-30"],
    ["016","107","Kelly Zhang","30","Female","49000","2018-04-01"],
    ["017","105","George Wang","34","Male","57000","2016-03-15"],
    ["018","104","Nancy Liu","29","Female","50000","2017-06-01"],
    ["019","103","Steven Chen","36","Male","62000","2015-08-01"],
    ["020","102","Grace Kim","32","Female","53000","2018-11-01"]
]

emp_schema = "employee_id string, department_id string, name string, age string, gender string, salary string, hire_date string"

In [ ]:
# Create emp DataFrame

emp = spark.createDataFrame(data=emp_data, schema=emp_schema)
emp.show()

In [ ]:
emp.rdd.getNumPartitions()

In [ ]:
# Show emp dataframe (ACTION)

emp.show()

In [ ]:
# Schema for emp

print(emp.schema)

In [ ]:
# Small Example for Schema
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
schema_string = "name string, age int"

schema_spark =  StructType([
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True)
])

In [ ]:
# Columns and expression
from pyspark.sql.functions import col, expr

emp["salary"]


In [ ]:
# SELECT columns
# select employee_id, name, age, salary from emp

emp_filtered = emp.select(col("employee_id"), expr("name"), emp.age, emp.salary)

In [ ]:
# SHOW Dataframe (ACTION)

emp_filtered.show()

In [ ]:
# Using expr for select
# select employee_id as emp_id, name, cast(age as int) as age, salary from emp_filtered

emp_casted = emp_filtered.select(expr("employee_id as emp_id"), emp.name, expr("cast(age as int) as age"), emp.salary)

In [ ]:
# SHOW Dataframe (ACTION)

emp_casted.show()

In [ ]:
emp_casted_1 = emp_filtered.selectExpr("employee_id as emp_id", "name", "cast(age as int) as age", "salary")

In [ ]:
emp_casted_1.show()

In [ ]:
emp_casted.printSchema()

In [ ]:
# Filter emp based on Age > 30
# select emp_id, name, age, salary from emp_casted where age > 30

emp_final = emp_casted.select("emp_id", "name", "age", "salary").where("age > 30")

In [ ]:
# SHOW Dataframe (ACTION)

emp_final.show()

In [ ]:
# Write the data back as CSV (ACTION)

emp_final.write.format("csv").save("data/output/2/emp.csv")

In [ ]:
# Bonus TIP

schema_str = "name string, age int"

from pyspark.sql.types import _parse_datatype_string

schema_spark = _parse_datatype_string(schema_str)

schema_spark

In [ ]:
import pandas as pd

In [ ]:
pd.read_csv("/content/MegaMart.csv")

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("Basic Transformation - I")
    .master("local[*]")
    .getOrCreate()
)

spark

In [ ]:
df = spark.read.format("csv").option("header", True).option("infoSchema", True).load("/content/MegaMart.csv")

In [ ]:
df.show()

In [ ]:
df.printSchema()

In [ ]:
from pyspark.sql.functions import to_date, col

order_date_fix = df.withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd"))

In [ ]:
df_totalprice = df.withColumn("total_price", col("quantity") * col("price_per_unit"))

In [ ]:
df_totalprice.show()

In [ ]:
df_totalprice.where(col("order_status") == "Delivered").show()

In [ ]:
df.select("payment_method").distinct().show()

In [ ]:
df_nareplace = df_totalprice.select("payment_method")

In [ ]:
from pyspark.sql.functions import sum
df_revcountbycat = df_totalprice.groupBy("product_category").agg(sum("total_price")).alias("total_revenue")

In [ ]:
df_revcountbycat.show()

In [ ]:
from pyspark.sql.functions import count, col, sum, isnan, when
df.select([
    sum(when(col(c).isNull() | isnan(col(c)),1).otherwise(0)).alias(c + "_null_count")
    for c in df.columns
]).show()

In [ ]:
missing_count = df.select([
    sum(when(col(c).isNull() | isnan(col(c)),1).otherwise(0)).alias(c + "_null_count")
    for c in df.columns
]).rdd.flatMap(lambda x: x).sum()

print("Total Missing Values: ", missing_count)

In [ ]:
from pyspark.sql.functions import desc

top_5_products = df_totalprice.groupBy("product_name").agg(sum("total_price").alias("total_revenue")).orderBy(desc("total_revenue")).limit(5)

top_5_products.show()

In [ ]:
from pyspark.sql.functions import avg

average_revenue_per_user = df_totalprice.groupBy("user_id").agg(avg("total_price").alias("average_revenue"))

average_revenue_per_user.show()

In [ ]:
from pyspark.sql.functions import col, count, desc

cancelled_orders = df.filter(col("order_status") == "Cancelled")
category_cancelled_counts = cancelled_orders.groupBy("product_category").agg(count("*").alias("cancelled_count"))
category_with_max_cancelled = category_cancelled_counts.orderBy(desc("cancelled_count")).limit(1)

category_with_max_cancelled.show()

In [ ]:
category_cancelled_counts.show()

In [ ]:
from pyspark.sql.functions import col, count, desc

cancelled_orders = df.filter(col("order_status") == "Cancelled")
category_cancelled_counts = cancelled_orders.groupBy("payment_method").agg(count("*").alias("cancelled_count"))
category_with_max_cancelled = category_cancelled_counts.orderBy(desc("cancelled_count")).limit(1)

print(category_cancelled_counts.show())
print(category_with_max_cancelled.show())

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType, MapType
from datetime import date

In [ ]:
spark = SparkSession.builder.appName("Practice").getOrCreate()

In [ ]:
schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("product", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("price", DoubleType(), True),
    StructField("order_date", DateType(), True),
    StructField("tags", StringType(), True), # For split() and regex examples
    StructField("details", StringType(), True), # For translate(), substring() examples
    StructField("features", StringType(), True), # For regexp_extract(), regexp_replace() examples
    StructField("product_attributes", MapType(StringType(), StringType()), True), # For MapType, explode, map_keys, map_values
    StructField("related_products", StringType(), True) # For collect_list, collect_set
])

In [ ]:
# Sample data
data = [
    (1, 101, "Laptop", 1, 1200.00, date(2023, 1, 15), "electronics,office", "Serial: A1B2C3D4", "user_123_laptop", {"color": "silver", "brand": "XYZ"}, "Keyboard,Mouse,Monitor"),
    (2, 102, "Mouse", 2, 25.50, date(2023, 1, 15), "electronics,accessory", "Model: M500", "user_456_mouse", {"color": "black", "wireless": "true"}, "Laptop,Monitor"),
    (3, 101, "Keyboard", 1, 75.00, date(2023, 1, 16), "electronics,accessory", "SKU: KB-101", "id_789_keyboard", {"layout": "US", "mechanical": "false"}, "Mouse,Monitor"),
    (4, 103, "Monitor", 1, 300.00, date(2023, 1, 16), "electronics,display", "DisplaySize: 27inch", "user_123_monitor", {"size": "27", "resolution": "1080p"}, "Laptop,Keyboard"),
    (5, 102, "Desk Chair", 1, 150.00, date(2023, 1, 17), "furniture,office", "Weight: 20kg", "user_456_chair", {"material": "mesh", "adjustable": "true"}, "Desk,Lamp"),
    (6, 104, "Lamp", 2, 35.00, date(2023, 1, 17), "furniture,lighting", "BulbType: LED", "id_abc_lamp", None, "Desk Chair,Desk"), # Example with None map
    (7, 101, "Laptop", 1, 1200.00, date(2023, 1, 18), "electronics,office", "Serial: E5F6G7H8", "user_123_laptop", {"color": "silver", "brand": "XYZ"}, "Keyboard,Mouse,Monitor"), # Duplicate order
    (8, 105, "Notebook", 5, 3.00, date(2023, 1, 18), "office,stationery", "Pages: 100", "user_xyz_notebook", {}, "Pen,Pencil"), # Example with empty map
    (9, 103, "Desk", 1, 250.00, date(2023, 1, 19), "furniture,office", "Material: Wood", "user_789_desk", {"size": "medium"}, "Desk Chair,Lamp"),
    (10, 104, "Pen", 10, 1.50, date(2023, 1, 19), "office,stationery", "InkColor: Blue", "id_def_pen", {"color": "blue"}, "Notebook,Pencil"),
    (11, 105, "Pencil", 12, 0.50, date(2023, 1, 20), "office,stationery", "LeadSize: 0.7mm", "user_xyz_pencil", {"lead": "0.7"}, "Notebook,Pen"),
    (12, 106, "Tablet", 1, 400.00, date(2023, 1, 20), "electronics", "Model: Tab-Pro", "user_abc_tablet", {"os": "Android"}, None), # Example with None related_products
    (13, 106, "Protector", 1, 15.00, date(2023, 1, 20), "electronics,accessory", "Type: Screen", "user_abc_protector", {}, ""), # Example with empty related_products
    # Added new rows
    (14, 101, "Mouse", 1, 25.50, date(2023, 1, 21), "electronics,accessory", "Model: M600", "user_101_mouse", {"color": "white", "wireless": "false"}, "Keyboard"),
    (15, 102, "Laptop", 1, 1100.00, date(2023, 1, 21), "electronics,office", "Serial: I9J10K11L12", "user_102_laptop", {"color": "black", "brand": "UVW"}, "Mouse,Monitor"),
    (16, 103, "Keyboard", 2, 70.00, date(2023, 1, 22), "electronics,accessory", "SKU: KB-202", "user_103_keyboard", {"layout": "UK", "mechanical": "true"}, "Mouse"),
    (17, 104, "Desk", 1, 220.00, date(2023, 1, 22), "furniture,office", "Material: Metal", "user_104_desk", {"size": "large"}, "Chair"),
    (18, 105, "Lamp", 1, 30.00, date(2023, 1, 23), "furniture,lighting", "BulbType: Incandescent", "user_105_lamp", None, "Desk"),
    (19, 106, "Notebook", 3, 2.50, date(2023, 1, 23), "office,stationery", "Pages: 150", "user_106_notebook", {}, "Pen,Pencil"),
    (20, 101, "Monitor", 1, 280.00, date(2023, 1, 24), "electronics,display", "DisplaySize: 24inch", "user_101_monitor", {"size": "24", "resolution": "1080p"}, "Laptop")
]

# Make sure to run the cell that initializes the SparkSession first.
practice_df = spark.createDataFrame(data, schema)

In [ ]:
print("Sample Practice DataFrame:")
practice_df.show(truncate=False)
practice_df.printSchema()

In [ ]:
from pyspark.sql.functions import explode, split, col
df_exploded_practice = practice_df.select(col("order_id"), explode(split(col("tags"),",")).alias("tag"))

In [ ]:
df_exploded_practice.show()

# Count how many orders are in each tag.


In [ ]:
df_exploded_practice.groupBy("tag").count().show()

In [ ]:
from pyspark.sql.functions import regexp_extract
practice_df.select(col("details"), regexp_extract(col("details"), r"Serial: (.*)", 1).alias("Serial_number")).show()

#or

practice_df.withColumn("Seial_number", regexp_extract(col("details"), r"Serial: (.*)", 1)).show()

In [ ]:
practice_df.select(col("features"), regexp_extract(col("features"), r"user_(.*)", 1).alias("User_id")).show()

In [ ]:
from pyspark.sql.functions import regexp_replace

practice_df = practice_df.withColumn("features",regexp_replace(col("features"), "^id_", "user_"))

In [ ]:
practice_df.show()

In [ ]:
practice_df.select(col("features"), regexp_extract(col("features"), r"user_(.*)", 1).alias("User_id")).show()

In [ ]:
from pyspark.sql.functions import translate
practice_df.select(col("details"), translate(col("details"), "aeiouAEIOU", "").alias("Vowels_removed")).show()

In [ ]:
from pyspark.sql.functions import substring
practice_df.withColumn("new_details", substring(col("details"), 1,3)).show()

In [ ]:
practice_df.withColumn("Fisrt_five", substring(col("features"), 1, 5)).show()

In [ ]:
from pyspark.sql.functions import concat_ws
practice_df.select(col("product"), col("quantity"), concat_ws("_", col("product"), col("quantity")).alias("product_quantity")).show()

In [ ]:
from pyspark.sql.functions import concat_ws
new_list = practice_df.withColumn("New_list", concat_ws(",", col("related_products")))
new_list.show()


In [ ]:
from pyspark.sql.functions import split, col

# Split the related_products_string into an array of strings
practice_df.select(col("related_products"), split(col("related_products"), ",").alias("related_products_list")).show()
